<a href="https://colab.research.google.com/github/lydiacyhung/114-2-ProgramingLanguage/blob/main/HW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位: date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1fc5laOITajG78TNIXouEP2ernmL9MKvi5VJb2jVWQyw/edit?gid=1003821083#gid=1003821083

In [2]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [3]:
import pandas as pd
# read data and put it in a dataframe
# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url('https://docs.google.com/spreadsheets/d/1fc5laOITajG78TNIXouEP2ernmL9MKvi5VJb2jVWQyw/edit?gid=1003821083#gid=1003821083')

In [4]:
# 從 gsheets 的 All-whiteboard-device 載入 sheets
gsheets = gc.open_by_url('https://docs.google.com/spreadsheets/d/1fc5laOITajG78TNIXouEP2ernmL9MKvi5VJb2jVWQyw/edit?gid=1003821083#gid=1003821083')
sheets = gsheets.worksheet('工作表1').get_all_values()
headers = sheets[0] if sheets else []
data_rows = sheets[1:] if sheets else []

# 確保 '交易類型' column exists. If not, add it.
if '交易類型' not in headers:
    # Find a suitable position to insert '交易類型', e.g., after '時間'
    try:
        insert_index = headers.index('時間') + 1
    except ValueError:
        insert_index = len(headers) # If '時間' not found, append at the end

    headers.insert(insert_index, '交易類型')
    for row in data_rows:
        # Ensure row has enough elements before inserting
        if len(row) < insert_index:
            row.extend([''] * (insert_index - len(row)))
        row.insert(insert_index, '') # Insert empty string for existing rows

# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(data_rows, columns=headers)
# 取得最前面的5筆資料
df.head()

,日期,交易類型,分類,品項,金額,付款人,地點,支付方式
0,2026-03-12,收入,薪水,家教,112,,,


In [5]:
import datetime

In [6]:
import datetime

# 讓使用者輸入資料
year_str = input("請輸入年份 (例如: 2026): ")
month_str = input("請輸入月份 (例如: 3): ")
day_str = input("請輸入日期 (例如: 12): ")

# 將分開輸入的年、月、日組合成一個完整的日期字串
date_str = f"{year_str}-{month_str.zfill(2)}-{day_str.zfill(2)}"

# 檢查日期格式是否正確
datetime.datetime.strptime(date_str, '%Y-%m-%d')


# 新增交易類型選擇
print("請選擇交易類型：")
print("1:支出 or 2:收入")
type_choice = input("請輸入數字 (1 或 2): ")

while type_choice not in ['1', '2']:
    print("輸入無效，請重新輸入 1 或 2。")
    type_choice = input("請輸入數字 (1 或 2): ")

transaction_type = '支出' if type_choice == '1' else '收入'

category = ""
item = ""
amount = ""
payer_or_source = ""
pay_method = ""

if transaction_type == '支出':
    category = input("請輸項目: ")
    item = input("請輸入品名: ")
    amount = input("請輸入金額: ")
    payer_or_source = input("請輸入付款人: ")
    pay_method = input("請輸入付款方式: ")
else: # 收入
    category = input("請輸入項目: ")
    item = input("請輸入名稱: ")
    amount = input("請輸入金額: ")



請輸入年份 (例如: 2026): 2026
請輸入月份 (例如: 3): 03
請輸入日期 (例如: 12): 25
請選擇交易類型：
1:支出 or 2:收入
請輸入數字 (1 或 2): 1
請輸項目: 午餐
請輸入品名: 麥當勞
請輸入金額: 145
請輸入付款人: 承
請輸入付款方式: 現金


In [7]:
# First, ensure df's columns are unique. This is the root cause of InvalidIndexError.
# Rename duplicate empty string columns.
new_df_columns = []
column_counts = {}
for col in df.columns:
    if col == '': # If column name is empty
        column_counts[''] = column_counts.get('', 0) + 1
        new_df_columns.append(f'Unnamed_{column_counts[""] - 1}' if column_counts[''] > 1 else 'Unnamed_0')
    elif col in column_counts: # If non-empty column is duplicated
        column_counts[col] += 1
        new_df_columns.append(f'{col}_{column_counts[col]-1}')
    else:
        column_counts[col] = 1
        new_df_columns.append(col)
df.columns = new_df_columns

# Now, create a new DataFrame for the input data, aligning with df's corrected columns
# The original df has '支付方式', but new_data uses '付款方式' in the prompt's comment.
# We will use '支付方式' to match the sheet.
new_record_data = {
    '日期': date_str,
    '交易類型': transaction_type, # 新增交易類型
    '分類': category,
    '品項': item,
    '金額': amount,
    '付款人': payer_or_source, # 統一使用 '付款人' 欄位來記錄付款人或收入來源
    '支付方式': pay_method # Correcting column name to match existing df
}

# Create a DataFrame from the new record data, ensuring it has all columns present in df
# and filling missing ones with NaN (or None)
new_data_aligned = pd.DataFrame([new_record_data])

# To ensure proper concatenation, it's best to create a new DataFrame with all columns
# from df and fill it with new_record_data where available.
final_new_data_df = pd.DataFrame(columns=df.columns)
for col in df.columns:
    if col in new_data_aligned.columns:
        final_new_data_df[col] = new_data_aligned[col]
    else:
        # If a column exists in df but not in new_data_aligned (like '地點', '備註', or the unnamed ones),
        # fill it with None for the new row.
        final_new_data_df[col] = None

# 使用 concat() 將新資料合併到舊的 df 中
df = pd.concat([df, final_new_data_df], ignore_index=True)

In [8]:
import numpy as np
# 將 DataFrame 中的 NaN 替換為 None，因為 gspread 不支援 NaN
df_cleaned = df.replace({np.nan: None})

# 將 DataFrame 轉換成列表的列表 (list of lists)，這是 gspread 支援的資料格式
data_to_write = df_cleaned.values.tolist()

# 第一步：取得工作表物件
worksheet = gsheets.worksheet('工作表1')

# 第二步：將資料寫入工作表
worksheet.append_rows(values=data_to_write, value_input_option='USER_ENTERED')
print("資料已成功寫入 Google 工作表！")

資料已成功寫入 Google 工作表！


In [9]:
df

,日期,交易類型,分類,品項,金額,付款人,地點,支付方式
0,2026-03-12,收入,薪水,家教,112,,,
1,2026-03-25,支出,午餐,麥當勞,145,承,None,現金


In [17]:
from google.colab import auth
from google.auth import default
auth.authenticate_user()
creds, project = default()
print(creds.service_account_email)

default


In [18]:
import gradio as gr
import pandas as pd
import datetime
import gspread
from google.colab import auth
from google.auth import default

SHEET_URL = "https://docs.google.com/spreadsheets/d/1fc5laOITajG78TNIXouEP2ernmL9MKvi5VJb2jVWQyw/edit?gid=1003821083#gid=1003821083"
WORKSHEET_NAME = "工作表1"

REQUIRED_COLUMNS = ["日期", "時間", "分類", "品項", "金額", "付款人"]

_auth_done = False
_gc = None
_ws = None


def _ensure_auth_and_ws():
    global _auth_done, _gc, _ws

    if not _auth_done:
        auth.authenticate_user()
        creds, _ = default()
        _gc = gspread.authorize(creds)
        _auth_done = True

    if _ws is None:
        gs = _gc.open_by_url(SHEET_URL)
        _ws = gs.worksheet(WORKSHEET_NAME)
        _ensure_headers()

    return _ws


def _ensure_headers():
    rows = _ws.get_all_values()

    if not rows:
        _ws.append_row(REQUIRED_COLUMNS, value_input_option="USER_ENTERED")
        return

    header = rows[0]

    if header != REQUIRED_COLUMNS:
        existing = {h: idx for idx, h in enumerate(header)}

        _ws.update("1:1", [REQUIRED_COLUMNS])

        if len(rows) > 1:
            new_rows = []
            for r in rows[1:]:
                mapping = {
                    col: (r[existing[col]] if col in existing and existing[col] < len(r) else "")
                    for col in REQUIRED_COLUMNS
                }
                new_rows.append([mapping[c] for c in REQUIRED_COLUMNS])

            _ws.resize(rows=1)
            _ws.append_rows(new_rows, value_input_option="USER_ENTERED")


def _read_df():
    ws = _ensure_auth_and_ws()
    values = ws.get_all_values()

    if not values:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    df = pd.DataFrame(values[1:], columns=values[0])

    if "金額" in df.columns:
        df["金額"] = pd.to_numeric(df["金額"], errors="coerce")

    return df


def add_record(date_str, category, item, amount, payer): # Removed time_str from parameters
    try:
        if not date_str:
            date_str = datetime.date.today().strftime("%Y-%m-%d")

        datetime.datetime.strptime(date_str, "%Y-%m-%d")

        # Removed time_str validation as it's no longer an input

        category = (category or "未分類").strip()
        item = (item or "未填").strip()
        payer = (payer or "本人").strip()

        try:
            amount_val = float(amount)
        except:
            return "❌ 金額格式錯誤，請輸入數字", pd.DataFrame()

        ws = _ensure_auth_and_ws()
        ws.append_row(
            [date_str, "", category, item, amount_val, payer], # Pass empty string for '時間'
            value_input_option="USER_ENTERED"
        )

        df = _read_df()
        preview_df = df.tail(10)

        msg = f"✅ 已成功新增：{date_str} {'' or ''}｜{category}｜{item}｜{amount_val}｜{payer}"
        return msg, preview_df

    except Exception as e:
        return f"❌ 新增失敗：{e}", pd.DataFrame()


def view_all_records():
    try:
        df = _read_df()
        if df.empty:
            return pd.DataFrame(columns=REQUIRED_COLUMNS)
        return df
    except Exception as e:
        return pd.DataFrame({"錯誤": [str(e)]})


with gr.Blocks(title="記帳小工具") as demo:
    gr.Markdown("""
    ## 🧾 記帳小工具
    這個版本先專注在「新增記帳」與「查看資料」。

    你可以：
    - 直接新增一筆支出資料到 Google Sheet
    - 查看目前所有記帳內容
    """)

    with gr.Tab("➕ 新增記帳"):
        with gr.Row():
            date_in = gr.Textbox(
                label="日期 YYYY-MM-DD",
                value=datetime.date.today().strftime("%Y-%m-%d")
            )

        with gr.Row():
            cat_in = gr.Textbox(
                label="分類",
                placeholder="例如：早餐 / 交通 / 日用品"
            )
            item_in = gr.Textbox(
                label="品項",
                placeholder="例如：便當 / 衣服 / 飲料"
            )

        with gr.Row():
            amt_in = gr.Textbox(
                label="金額",
                placeholder="請輸入數字"
            )
            payer_in = gr.Textbox(
                label="付款人",
                placeholder="例如：自己 / 媽媽 / 小明"
            )

        add_btn = gr.Button("新增到工作表")
        add_msg = gr.Markdown()
        preview_df = gr.Dataframe(label="最近 10 筆資料", interactive=False)

        add_btn.click(
            fn=add_record,
            inputs=[date_in,cat_in, item_in, amt_in, payer_in],
            outputs=[add_msg, preview_df]
        )

    with gr.Tab("📒 檢視全部資料"):
        view_btn = gr.Button("讀取全部資料")
        view_df = gr.Dataframe(label="全部記帳資料", interactive=False)

        view_btn.click(
            fn=view_all_records,
            inputs=[],
            outputs=[view_df]
        )

demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>